In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('global_supply_chain_risk_2026.csv', sep=',')


padrao_texto = ['Origin_Port', 'Destination_Port', 'Transport_Mode', 'Product_Category', 'Weather_Condition']
df[padrao_texto] = df[padrao_texto].astype('string')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df = df.drop(columns='Shipment_ID')

df.rename(columns={'Date' : 'Data',
                   'Origin_Port' : 'Ponto_Origem',
                   'Destination_Port' : 'Ponto_Chegada',
                   'Transport_Mode' : 'Modo_Transporte',
                   'Product_Category' : 'Categoria_do_Produto',
                   'Distance_km' : 'Distancia_KM',
                   'Weight_MT' : 'Peso_Remessa_Tonelada',
                   'Fuel_Price_Index' : 'Indice_Preco_Combustivel',
                   'Geopolitical_Risk_Score' : 'Pontuaçao_Risco_Geopolitico',
                   'Weather_Condition' : 'Condiçao_Climatica',
                   'Carrier_Reliability_Score' : 'Indice_Confiabilidade_Transportadora',
                   'Lead_Time_Days' : 'Prazo_Entrega_Dias',
                   'Disruption_Occurred' : 'Problema_Logistico'},inplace=True)

df


# ===========================
# Taxa de Problemas Logísticos
# ===========================
df_contagem_problemas_logistico = (df['Problema_Logistico'].agg(
                    contagem_total = 'count',
                    contagem_problemas = 'sum',
                )
                .to_frame().T
                .assign(
                    contagem_sem_problem = lambda x: x['contagem_total'] - x['contagem_problemas']
                ))


df_representacao_problema = (df_contagem_problemas_logistico['contagem_problemas'] / df_contagem_problemas_logistico['contagem_total'] * 100).round(2)

display(df_contagem_problemas_logistico)
display(df_representacao_problema)    

# =====================================
# Grafico (Taxa de Problemas Logísticos)
# =====================================
problemas = 3063
sem_problema = 5000 - 3063
labels = ['Com Problemas logisticos', 'Sem Problemas logisticos']
vals = [problemas, sem_problema]
plt.figure()
plt.pie(vals, autopct='%.2f%%')
plt.title('Índice de Ineficiência Logística')
plt.legend(labels)
plt.tight_layout()
plt.show()



# ================================
# DataFrame de Problemas Logísticos
# ================================
df_analise_problemas_gerais = (df.query('Problema_Logistico == 1'))
display(df_analise_problemas_gerais)



# ========================================
# Análise de Problemas por Ponto de Origem
# ========================================
df_problemas_ponto_origem = (df_analise_problemas_gerais.groupby('Ponto_Origem', as_index=False)
                            .agg(
                                 Problema_Logistico = ('Problema_Logistico','sum'),
                                 Pontuaçao_Media_Risco_Geopolitico = ('Pontuaçao_Risco_Geopolitico', 'mean')
                            )
                            .sort_values(by= 'Problema_Logistico', ascending=False)
                            .reset_index(drop=True)
                            .assign(
                                 representacao_em_porcentagem = lambda x: (x['Problema_Logistico'] / x['Problema_Logistico'].sum() * 100).round(2)
                            )
)
display(df_problemas_ponto_origem)

# =================================================
# Grafico (Análise de Problemas por Ponto de Origem)
# =================================================
plt.figure(figsize=(8,5))  
grafico_barras_ponto_de_origem = plt.bar(                 
    df_problemas_ponto_origem['Ponto_Origem'],            
    df_problemas_ponto_origem['Problema_Logistico']       
)

for barra in grafico_barras_ponto_de_origem:            
    altura = barra.get_height()                           
    plt.text(                                            
        barra.get_x() + barra.get_width()/2,             
        altura,                                           
        int(altura),                                      
        ha = 'center',                                   
        va = 'bottom'                                    
    )
plt.tight_layout()                                     
plt.title('Gargalos Logísticos por Ponto de origem')
plt.xlabel('Ponto de origem')
plt.ylabel('Quantidade de problemas')
plt.show()


# ========================================
# Análise de Problemas por Ponto de Chegada
# ========================================
df_problemas_ponto_chegada = (df_analise_problemas_gerais.groupby('Ponto_Chegada', as_index=False)
                            .agg(
                                Problema_Logistico = ('Problema_Logistico', 'sum'),
                                Pontuaçao_Media_Risco_Geopolitico = ('Pontuaçao_Risco_Geopolitico', 'mean')
                            )
                            .sort_values(by= 'Problema_Logistico', ascending=False)
                            .reset_index(drop=True)
                            .assign(
                                 Representacao_em_Porcentagem = lambda x: (x['Problema_Logistico'] / x['Problema_Logistico'].sum() * 100).round(2)
                                )
)
display(df_problemas_ponto_chegada)

# ==================================================
# Grafico (Análise de Problemas por Ponto de Chegada)
# ==================================================
plt.figure(figsize=(8,5))
grafico_barras_ponto_de_chegada = plt.bar(
    df_problemas_ponto_chegada['Ponto_Chegada'],
    df_problemas_ponto_chegada['Problema_Logistico']
)

for barra_2 in grafico_barras_ponto_de_chegada:
    altura_2 = barra_2.get_height()
    plt.text(
        barra_2.get_x() + barra_2.get_width()/2,
        altura_2,
        int(altura_2),
        ha = 'center',
        va = 'bottom'
    )
plt.tight_layout()
plt.title('Gargalos Logísticos por Ponto de Chegada')
plt.xlabel('Pontos de chegada')
plt.ylabel('Quantidade de problemas')
plt.show()


# =========================================
# Top 10 Rotas com Mais Problemas Logísticos
# =========================================
df_top_10_rotas_com_problemas = (
                         df_analise_problemas_gerais.groupby(['Ponto_Origem', 'Ponto_Chegada'], as_index=False)
                         .agg(
                              Problema_Logistico = ('Problema_Logistico', 'sum'),
                              Distancia_Media_KM = ('Distancia_KM', 'mean'),
                              Peso_Medio_Remessa_Tonelada = ('Peso_Remessa_Tonelada', 'mean'),
                              Indice_Medio_Preco_Combustivel = ('Indice_Preco_Combustivel', 'mean'),
                              Indice_Medio_Confiabilidade_Transportadora = ('Indice_Confiabilidade_Transportadora', 'mean'),
                              Media_Prazo_Entrega_Dias = ('Prazo_Entrega_Dias', 'mean'),
                              Pontuacao_Media_Risco_Geopolitico = ('Pontuaçao_Risco_Geopolitico', 'mean')
                         )
                         .sort_values(by=['Problema_Logistico'], ascending=False)
                         .reset_index(drop=True)
                         .head(10)
)
display(df_top_10_rotas_com_problemas)

# ==================================================
# Grafico (Top 10 Rotas com Mais Problemas Logísticos)
# ==================================================
plt.figure(figsize=(8,6))

barra = plt.bar(
    df_top_10_rotas_com_problemas['Ponto_Origem'] + '→' + df_top_10_rotas_com_problemas['Ponto_Chegada'],
    df_top_10_rotas_com_problemas['Problema_Logistico']
)

for barras in barra:
    altura_3 = barras.get_height()
    plt.text(
        barras.get_x() + barras.get_width()/2,
        altura_3,
        int(altura_3),
        ha = 'center',
        va = 'bottom'
    )

plt.title('Top 10 Trechos com Maior Atrito Logístico')
plt.xlabel('Rotas')
plt.ylabel('Quantidade de Problemas Logístico')
plt.xticks(rotation = 45, ha = 'right')
plt.tight_layout()
plt.show()


# =========================================
# Problemas Logísticos por Modo de Transporte
# =========================================
df_modo_transporte_problema = (
    df_analise_problemas_gerais.groupby('Modo_Transporte', as_index = False)
    .agg(
        Problemas_Logistico = ('Problema_Logistico', 'sum'),
        Media_diastancia_KM = ('Distancia_KM', 'mean')
    )
    .sort_values(by='Problemas_Logistico', ascending=True)
    .assign(
        Representacao_porcentagem = lambda x: (x['Problemas_Logistico'] / x['Problemas_Logistico'].sum() * 100).round(2),
        Média_Variacao_percentual = lambda x: (x['Problemas_Logistico'].std() / x['Problemas_Logistico'].mean()).round(2)
    )
)
display(df_modo_transporte_problema)

# ===================================================
# Grafico (Problemas Logísticos por Modo de Transporte)
# ===================================================
labelss = df_modo_transporte_problema['Modo_Transporte'].to_list()
valss = df_modo_transporte_problema['Problemas_Logistico']
nomes_modo_transporte = ['Aéreo',  'Marítimo', 'Ferroviário', 'Rodoviário']

plt.Figure(figsize=(8,5))
plt.pie(valss , autopct='%.2f%%')
plt.legend(nomes_modo_transporte)
plt.title('Distribuição de Incidentes por Modalidade de Transporte')
plt.tight_layout()
plt.show()



# =================================
# Problemas por Categoria de Produto
# =================================
df_categorias_problemas = (df_analise_problemas_gerais
                           .groupby('Categoria_do_Produto', as_index=False)['Problema_Logistico'].sum()
                           .sort_values(by='Problema_Logistico', ascending=True)
                           .reset_index(drop=True)
                           .assign(
                               representacao_porcent = lambda x: (x['Problema_Logistico']/ x['Problema_Logistico'].sum() * 100).round(2),
                               Média_variacao_percentual = lambda x: (x['Problema_Logistico'].std() / x['Problema_Logistico'].mean()).round(2)
                           ))
display(df_categorias_problemas)

# ===========================================
# Grafico (Problemas por Categoria de Produto)
# ===========================================
labelsss = df_categorias_problemas['Categoria_do_Produto']
valsss = df_categorias_problemas['Problema_Logistico']
categorias = ['Têxteis', 'Farmacêuticos', 'Perecíveis', 'Eletrônicos', 'Automotivo']

plt.figure(figsize=(8,5))
plt.pie(valsss, autopct='%.2f%%')
plt.legend(categorias)
plt.title('Incidência de Problemas por Categoria de Produto')
plt.tight_layout()
plt.show()


# =====================================================
# Análise dos Casos Sem Problemas Logísticos (Comparativo)
# =====================================================
df_analise_sem_problemas = (df.query('Problema_Logistico == 0')
                            .groupby(['Ponto_Origem', 'Ponto_Chegada'], as_index=False)
                            .agg(
                                Distancia_Media_KM = ('Distancia_KM', 'mean'),
                                Peso_Medio_Remessa_Tonelada = ('Peso_Remessa_Tonelada', 'mean'),
                                Indice_Medio_Preco_Combustivel = ('Indice_Preco_Combustivel', 'mean'),
                                Pontuacao_Media_Risco_Geopolitico = ('Pontuaçao_Risco_Geopolitico', 'mean'),
                                Indice_Medio_Confiabilidade_Transportadora = ('Indice_Confiabilidade_Transportadora', 'mean'),
                                Media_Prazo_Entrega_Dias = ('Prazo_Entrega_Dias', 'mean')
                            )
                            .sort_values(by='Distancia_Media_KM', ascending=False)
)
display(df_analise_sem_problemas.head(10))

